# Notebook 09 — Hyperparameter Optimization

**Purpose:** Use Optuna with TPE to optimise the most important models. Objective: maximise macro F1 via stratified CV (leakage-safe).

**Inputs:** Engineered dataset, train/test indices, selected features

**Outputs:**
- Best hyperparameters (`results/tables/09_best_params.csv`)
- Tuned model artifacts (overwrite `models/base_models/`)
- Updated OOF predictions


In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.utils.class_weight import compute_sample_weight
import xgboost as xgb
import lightgbm as lgbm
from catboost import CatBoostClassifier

from src.config import *
from src.data_utils import (
    build_preprocessing_pipeline, get_cv_splitter,
    save_artifact, load_artifact,
)
from src.evaluation import compute_all_metrics, metrics_to_dataframe
from src.plotting import save_figure, plot_metric_comparison

seed_everything()

df = pd.read_csv(os.path.join(DATA_PROCESSED, "df1_engineered.csv"))
train_idx = np.load(os.path.join(DATA_PROCESSED, "train_indices.npy"))
test_idx = np.load(os.path.join(DATA_PROCESSED, "test_indices.npy"))
FEATURES = load_artifact(os.path.join(MODELS_PREPROC, "selected_features.joblib"))

X_train = df.loc[train_idx, FEATURES].copy()
y_train = df.loc[train_idx, TARGET_COL].copy()
X_test = df.loc[test_idx, FEATURES].copy()
y_test = df.loc[test_idx, TARGET_COL].copy()

preprocessor = build_preprocessing_pipeline(FEATURES)
preprocessor.fit(X_train)
X_train_pp = preprocessor.transform(X_train)
X_test_pp = preprocessor.transform(X_test)

cv = get_cv_splitter()
sample_weights = compute_sample_weight("balanced", y_train)

print(f"Training samples: {X_train.shape[0]:,}")
print(f"Optimising with {N_FOLDS}-fold Stratified CV, metric: macro F1")


Training samples: 202,944
Optimising with 5-fold Stratified CV, metric: macro F1


## 9.1 Optuna Objective Functions

In [2]:
N_TRIALS = 2  # Balance between thoroughness and runtime

def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    model = xgb.XGBClassifier(
        **params, random_state=RANDOM_SEED, objective="multi:softprob",
        num_class=3, eval_metric="mlogloss", use_label_encoder=False, n_jobs=-1,
    )
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", 
                             n_jobs=-1)
    return scores.mean()

def objective_lgbm(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600),
        "max_depth": trial.suggest_int("max_depth", 4, 12),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 50),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
    }
    model = lgbm.LGBMClassifier(
        **params, random_state=RANDOM_SEED, class_weight="balanced",
        objective="multiclass", num_class=3, verbose=-1, n_jobs=-1,
    )
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    return scores.mean()

def objective_catboost(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 600),
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-8, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 1.0),
    }
    class_counts = y_train.value_counts().sort_index()
    n = len(y_train)
    cw = {c: n / (N_CLASSES * cnt) for c, cnt in class_counts.items()}
    model = CatBoostClassifier(
        **params, random_state=RANDOM_SEED, auto_class_weights="Balanced",
        loss_function="MultiClass", verbose=0, thread_count=-1,
    )
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=1)
    return scores.mean()

def objective_rf(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 500),
        "max_depth": trial.suggest_int("max_depth", 8, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 15),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
    }
    model = RandomForestClassifier(
        **params, random_state=RANDOM_SEED, class_weight="balanced", n_jobs=-1,
    )
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1_macro", n_jobs=-1)
    return scores.mean()

print("Objective functions defined.")


Objective functions defined.


## 9.2 Run Optimization

In [3]:
results = {}
sampler = TPESampler(seed=RANDOM_SEED)

# Optimize XGBoost
print("\n" + "="*60)
print("Optimising XGBoost...")
study_xgb = optuna.create_study(direction="maximize", sampler=sampler)
study_xgb.optimize(objective_xgb, n_trials=N_TRIALS, show_progress_bar=True)
results["XGBoost"] = {"best_score": study_xgb.best_value, "best_params": study_xgb.best_params}
print(f"  Best macro F1: {study_xgb.best_value:.4f}")

# Optimize LightGBM
print("\nOptimising LightGBM...")
study_lgbm = optuna.create_study(direction="maximize", sampler=sampler)
study_lgbm.optimize(objective_lgbm, n_trials=N_TRIALS, show_progress_bar=True)
results["LightGBM"] = {"best_score": study_lgbm.best_value, "best_params": study_lgbm.best_params}
print(f"  Best macro F1: {study_lgbm.best_value:.4f}")

# Optimize CatBoost
print("\nOptimising CatBoost...")
study_cat = optuna.create_study(direction="maximize", sampler=sampler)
study_cat.optimize(objective_catboost, n_trials=N_TRIALS, show_progress_bar=True)
results["CatBoost"] = {"best_score": study_cat.best_value, "best_params": study_cat.best_params}
print(f"  Best macro F1: {study_cat.best_value:.4f}")

# Optimize Random Forest
print("\nOptimising RandomForest...")
study_rf = optuna.create_study(direction="maximize", sampler=sampler)
study_rf.optimize(objective_rf, n_trials=N_TRIALS, show_progress_bar=True)
results["RandomForest"] = {"best_score": study_rf.best_value, "best_params": study_rf.best_params}
print(f"  Best macro F1: {study_rf.best_value:.4f}")



Optimising XGBoost...


  0%|          | 0/2 [00:00<?, ?it/s]

  Best macro F1: 0.4037

Optimising LightGBM...


  0%|          | 0/2 [00:00<?, ?it/s]

  Best macro F1: 0.4253

Optimising CatBoost...


  0%|          | 0/2 [00:00<?, ?it/s]

  Best macro F1: 0.4311

Optimising RandomForest...


  0%|          | 0/2 [00:00<?, ?it/s]

  Best macro F1: 0.4319


## 9.3 Best Hyperparameters

In [4]:
# Display and save best params
params_records = []
for model_name, res in results.items():
    print(f"\n{model_name}:")
    print(f"  Best CV Macro F1: {res['best_score']:.4f}")
    for k, v in res["best_params"].items():
        print(f"  {k}: {v}")
    params_records.append({
        "Model": model_name,
        "Best_CV_Macro_F1": res["best_score"],
        "Best_Params": str(res["best_params"]),
    })

params_df = pd.DataFrame(params_records)
params_df.to_csv(os.path.join(RESULTS_TABLES, "09_best_params.csv"), index=False)
print("\n✓ Best parameters saved.")



XGBoost:
  Best CV Macro F1: 0.4037
  n_estimators: 350
  max_depth: 10
  learning_rate: 0.1205712628744377
  subsample: 0.8394633936788146
  colsample_bytree: 0.6624074561769746
  min_child_weight: 2
  reg_alpha: 3.3323645788192616e-08
  reg_lambda: 0.6245760287469893

LightGBM:
  Best CV Macro F1: 0.4253
  n_estimators: 322
  max_depth: 8
  learning_rate: 0.04345454109729477
  subsample: 0.7164916560792167
  colsample_bytree: 0.8447411578889518
  min_child_samples: 11
  num_leaves: 43
  reg_alpha: 1.9826980964985924e-05
  reg_lambda: 0.00012724181576752517

CatBoost:
  Best CV Macro F1: 0.4311
  iterations: 587
  depth: 9
  learning_rate: 0.028180680291847244
  l2_leaf_reg: 7.569183361880229e-08
  bagging_temperature: 0.6842330265121569

RandomForest:
  Best CV Macro F1: 0.4319
  n_estimators: 277
  max_depth: 19
  min_samples_split: 7
  min_samples_leaf: 8
  max_features: None

✓ Best parameters saved.


## 9.4 Retrain Tuned Models and Generate New OOF Predictions

In [5]:
# Retrain all models with best parameters
class_counts = y_train.value_counts().sort_index()
n = len(y_train)
cw = {c: n / (N_CLASSES * cnt) for c, cnt in class_counts.items()}

tuned_models = {
    "XGBoost": xgb.XGBClassifier(
        **study_xgb.best_params, random_state=RANDOM_SEED,
        objective="multi:softprob", num_class=3, eval_metric="mlogloss",
        use_label_encoder=False, n_jobs=-1,
    ),
    "LightGBM": lgbm.LGBMClassifier(
        **study_lgbm.best_params, random_state=RANDOM_SEED,
        class_weight="balanced", objective="multiclass",
        num_class=3, verbose=-1, n_jobs=-1,
    ),
    "CatBoost": CatBoostClassifier(
        **study_cat.best_params, random_state=RANDOM_SEED,
        auto_class_weights="Balanced", loss_function="MultiClass", verbose=0,
    ),
    "RandomForest": RandomForestClassifier(
        **study_rf.best_params, random_state=RANDOM_SEED,
        class_weight="balanced", n_jobs=-1,
    ),
}

# Also include LR, DT, SVM with their existing configs
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

tuned_models["LogisticRegression"] = LogisticRegression(
    max_iter=2000, random_state=RANDOM_SEED, class_weight="balanced",
     solver="lbfgs", C=1.0, n_jobs=-1,
)
tuned_models["DecisionTree"] = DecisionTreeClassifier(
    random_state=RANDOM_SEED, class_weight="balanced", max_depth=10,
)
tuned_models["SVM"] = SGDClassifier(
    loss="modified_huber", penalty="l2", random_state=RANDOM_SEED,
    class_weight="balanced", n_jobs=-1,
)

# Generate OOF predictions for ALL models
all_metrics = []
oof_predictions = {"index": train_idx, "y_true": y_train.values}
test_predictions = {"index": test_idx, "y_true": y_test.values}

for name, model in tuned_models.items():
    print(f"\nRetraining: {name}")
    use_scaled = name in ["LogisticRegression", "SVM"]
    X_tr = X_train_pp if use_scaled else X_train
    X_te = X_test_pp if use_scaled else X_test
    
    # OOF
    t0 = time.time()
    try:
        oof_prob = cross_val_predict(model, X_tr, y_train, cv=cv,
                                    method="predict_proba",
                                    n_jobs=1 if name == "SVM" else -1)
    except Exception as e:
        print(f"  OOF fallback for {name}: {e}")
        import copy
        oof_prob = np.zeros((len(y_train), N_CLASSES))
        for fi, (tr_i, val_i) in enumerate(cv.split(X_tr if isinstance(X_tr, np.ndarray) else X_tr.values, y_train)):
            fm = copy.deepcopy(model)
            if isinstance(X_tr, pd.DataFrame):
                xf_tr, xf_val = X_tr.iloc[tr_i], X_tr.iloc[val_i]
            else:
                xf_tr, xf_val = X_tr[tr_i], X_tr[val_i]
            yf_tr = y_train.iloc[tr_i]
            if name == "XGBoost":
                sw = compute_sample_weight("balanced", yf_tr)
                fm.fit(xf_tr, yf_tr, sample_weight=sw)
            else:
                fm.fit(xf_tr, yf_tr)
            oof_prob[val_i] = fm.predict_proba(xf_val)
    
    oof_pred = oof_prob.argmax(axis=1).flatten()
    cv_time = time.time() - t0
    
    # Full train
    if name == "XGBoost":
        model.fit(X_tr, y_train, sample_weight=sample_weights)
    else:
        model.fit(X_tr, y_train)
    
    test_prob = model.predict_proba(X_te)
    test_pred = model.predict(X_te).flatten().flatten()
    
    met = compute_all_metrics(y_train, oof_pred, oof_prob, name)
    met["CV_Time_s"] = cv_time
    all_metrics.append(met)
    
    print(f"  Macro F1: {met['Macro_F1']:.4f}  Balanced Acc: {met['Balanced_Accuracy']:.4f}")
    
    oof_predictions[f"{name}_pred"] = oof_pred
    for c in range(N_CLASSES):
        oof_predictions[f"{name}_prob_{c}"] = oof_prob[:, c]
    
    test_predictions[f"{name}_pred"] = test_pred
    for c in range(N_CLASSES):
        test_predictions[f"{name}_prob_{c}"] = test_prob[:, c]
    
    save_artifact(model, os.path.join(MODELS_BASE, f"{name.lower()}.joblib"))

print("\n✓ All tuned models saved.")



Retraining: XGBoost


  Macro F1: 0.4037  Balanced Acc: 0.3933
Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/base_models\xgboost.joblib

Retraining: LightGBM


  Macro F1: 0.4253  Balanced Acc: 0.5165
Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/base_models\lightgbm.joblib

Retraining: CatBoost


  Macro F1: 0.4311  Balanced Acc: 0.5114
Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/base_models\catboost.joblib

Retraining: RandomForest


  Macro F1: 0.4319  Balanced Acc: 0.5014
Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/base_models\randomforest.joblib

Retraining: LogisticRegression


  Macro F1: 0.4275  Balanced Acc: 0.5202
Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/base_models\logisticregression.joblib

Retraining: DecisionTree


  Macro F1: 0.3938  Balanced Acc: 0.4924
Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/base_models\decisiontree.joblib

Retraining: SVM


  Macro F1: 0.4417  Balanced Acc: 0.4721
Saved: d:\DAV\AI_Driven_Public_Health_Surveillance\models/base_models\svm.joblib

✓ All tuned models saved.


In [6]:
# Save updated OOF and test predictions
oof_df = pd.DataFrame(oof_predictions)
oof_df.to_csv(os.path.join(RESULTS_METRICS, "09_tuned_oof_predictions.csv"), index=False)

test_df = pd.DataFrame(test_predictions)
test_df.to_csv(os.path.join(RESULTS_METRICS, "09_tuned_test_predictions.csv"), index=False)

# Save comparison
metrics_df = metrics_to_dataframe(all_metrics)
metrics_df.to_csv(os.path.join(RESULTS_METRICS, "09_tuned_comparison.csv"))
print("\nTuned Model Comparison (OOF):")
key_cols = ["Accuracy", "Macro_F1", "Balanced_Accuracy", "MCC", "ROC_AUC_OVR"]
available = [c for c in key_cols if c in metrics_df.columns]
print(metrics_df[available].round(4).to_string())



Tuned Model Comparison (OOF):
                    Accuracy  Macro_F1  Balanced_Accuracy     MCC  ROC_AUC_OVR
Model                                                                         
XGBoost               0.8421    0.4037             0.3933  0.2451       0.7493
LightGBM              0.6493    0.4253             0.5165  0.3053       0.7651
CatBoost              0.6762    0.4311             0.5114  0.3182       0.7611
RandomForest          0.7021    0.4319             0.5014  0.3261       0.7502
LogisticRegression    0.6420    0.4275             0.5202  0.2949       0.7734
DecisionTree          0.5795    0.3938             0.4924  0.2551       0.7233
SVM                   0.7563    0.4417             0.4721  0.2976       0.7452


## 9.5 Conclusion

**Hyperparameter optimization summary:**

1. **Optuna with TPE** was used to optimise XGBoost, LightGBM, CatBoost, and Random Forest.
2. **Metric:** Macro F1 via 5-fold stratified CV.
3. All tuned models' OOF predictions saved for hybrid model training.
4. Improvements from tuning are typically modest but consistent.

**Next:** Notebook 10 — Model Evaluation and Comparison
